# Adversarial Training (PGD-AT) for the Keyword Spotter

The audio counterpart of `defenses/01_adversarial_training.ipynb` in the companion vision repo
(Madry et al., 2018): fine-tune the base classifier on PGD-generated adversarial waveforms
instead of clean ones, so the model learns a decision boundary that already accounts for the
kind of perturbation `attacks/whitebox/02_PGD.ipynb` exploits. Same recipe, same honesty about what to
expect: with a small model and a handful of epochs, this is a demonstration that the
before/after pipeline works end to end, not a claim of state-of-the-art robustness.


In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import copy
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)

def _load_wav(path, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, format=None, buffer_size=4096, backend=None):
    """Replaces torchaudio's TorchCodec-based loader (needs a separately installed FFmpeg, not
    available here) with soundfile, which reads WAV directly with no extra system dependencies."""
    data, sample_rate = sf.read(path, dtype='float32')
    waveform = torch.from_numpy(data)
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(1)
    if channels_first:
        waveform = waveform.transpose(0, 1)
    return waveform, sample_rate

torchaudio.load = _load_wav

### Setup: Model and Data
Same `KeywordCNN` architecture and dataset loading as `models/01_train_keyword_spotter.ipynb`.
`baseline_model` is loaded from the saved weights and left untouched; `at_model` is a separate
copy that gets fine-tuned on adversarial examples, so both can be compared against the exact
same attacks afterwards.

In [2]:
LABELS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
LABEL_TO_IDX = {label: i for i, label in enumerate(LABELS)}
SAMPLE_RATE = 16000

mel_spectrogram = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=64, n_fft=400, hop_length=160)
to_db = torchaudio.transforms.AmplitudeToDB()

def waveform_to_spectrogram(waveform):
    return to_db(mel_spectrogram(waveform))

class KeywordCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, spectrogram):
        x = self.pool(F.relu(self.bn1(self.conv1(spectrogram))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).flatten(1)
        return self.fc(x)

baseline_model = KeywordCNN(num_classes=len(LABELS))
baseline_model.load_state_dict(torch.load('../models/keyword_spotter.pth', map_location='cpu'))
baseline_model.eval()

at_model = copy.deepcopy(baseline_model)
print("Loaded baseline model, cloned for adversarial training")

Loaded baseline model, cloned for adversarial training


In [3]:
MAX_PER_CLASS = {'training': 150, 'testing': 40}

class KeywordSubset(Dataset):
    def __init__(self, subset):
        full = torchaudio.datasets.SPEECHCOMMANDS(root='../models/speech_commands_data', download=True, subset=subset)
        limit = MAX_PER_CLASS[subset]
        counts = {label: 0 for label in LABELS}
        self.items = []
        for i in range(len(full)):
            _, _, label, *_ = full.get_metadata(i)
            if label not in LABEL_TO_IDX or counts[label] >= limit:
                continue
            counts[label] += 1
            waveform, sr, label, *_ = full[i]
            self.items.append((waveform, label))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        waveform, label = self.items[idx]
        if waveform.shape[1] < SAMPLE_RATE:
            waveform = F.pad(waveform, (0, SAMPLE_RATE - waveform.shape[1]))
        else:
            waveform = waveform[:, :SAMPLE_RATE]
        return waveform, LABEL_TO_IDX[label]

os.makedirs('../models/speech_commands_data', exist_ok=True)
train_set = KeywordSubset('training')
test_set = KeywordSubset('testing')
print(f"Training: {len(train_set)}, Testing: {len(test_set)} clips")

Training: 1500, Testing: 400 clips


### PGD-AT Training Loop
For every batch, generate a 5-step PGD adversarial version at $\varepsilon = 0.02$ (mid-range in
`attacks/whitebox/02_PGD.ipynb`'s sweep) and train on the average of the clean and adversarial loss, the
"mixed-batch" variant of PGD-AT common in practice: training purely on adversarial examples with
too high a learning rate on a small, not-yet-well-converged base model turned out to be unstable
here (a first attempt collapsed clean accuracy to 19.8%, barely above the 10% chance rate, while
barely denting FGSM/PGD ASR), so the loss keeps a foot on clean performance while still exposing
the model to worst-case perturbations every step. 5 inner PGD steps rather than the usual 10 keeps
a training epoch tractable on CPU, the same kind of reduced-scale tradeoff documented throughout
this repo family.

In [4]:
def pgd_generate(model, waveforms, labels, epsilon, iters=5):
    alpha = epsilon / (iters / 2.0)
    adv = waveforms.clone()
    for _ in range(iters):
        adv = adv.clone().requires_grad_(True)
        logits = model(waveform_to_spectrogram(adv))
        loss = F.cross_entropy(logits, labels)
        grad = torch.autograd.grad(loss, adv)[0]
        adv = adv + alpha * grad.sign()
        perturbation = torch.clamp(adv - waveforms, -epsilon, epsilon)
        adv = torch.clamp(waveforms + perturbation, -1.0, 1.0).detach()
    return adv

AT_EPSILON = 0.02
BATCH_SIZE = 32
AT_EPOCHS = 5

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.Adam(at_model.parameters(), lr=2e-4)

for epoch in range(AT_EPOCHS):
    at_model.train()
    total_loss = 0.0
    for waveforms, labels in train_loader:
        at_model.eval()  # BatchNorm uses eval-mode running stats while generating the attack
        adv_waveforms = pgd_generate(at_model, waveforms, labels, AT_EPSILON)
        at_model.train()

        optimizer.zero_grad()
        clean_logits = at_model(waveform_to_spectrogram(waveforms))
        adv_logits = at_model(waveform_to_spectrogram(adv_waveforms))
        loss = 0.5 * F.cross_entropy(clean_logits, labels) + 0.5 * F.cross_entropy(adv_logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * len(labels)

    print(f"Epoch {epoch + 1}/{AT_EPOCHS}, mean training loss: {total_loss / len(train_set):.4f}")

at_model.eval()
torch.save(at_model.state_dict(), 'keyword_spotter_pgdat.pth')
print("Saved defenses/keyword_spotter_pgdat.pth")

C:\Users\frang\AppData\Local\Temp\ipykernel_13944\437446597.py:35: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  total_loss += float(loss) * len(labels)


Epoch 1/5, mean training loss: 6.4729


Epoch 2/5, mean training loss: 3.5462


Epoch 3/5, mean training loss: 2.8387


Epoch 4/5, mean training loss: 2.6241


Epoch 5/5, mean training loss: 2.5080
Saved defenses/keyword_spotter_pgdat.pth


### Evaluation: Clean Accuracy and Robustness, Before and After
The same 10 held-out test clips attacked in `attacks/whitebox/01_FGSM.ipynb` and `attacks/whitebox/02_PGD.ipynb`, run
against both `baseline_model` and `at_model`, at the same $\varepsilon$ used during training.

In [5]:
@torch.no_grad()
def clean_accuracy(model, dataset):
    correct = 0
    for waveform, label_idx in dataset:
        logits = model(waveform_to_spectrogram(waveform.unsqueeze(0)))
        pred = int(torch.argmax(logits, dim=-1))
        correct += int(pred == label_idx)
    return correct / len(dataset)

def fgsm_attack(model, waveform, true_label_idx, epsilon):
    waveform = waveform.clone().requires_grad_(True)
    logits = model(waveform_to_spectrogram(waveform.unsqueeze(0)))
    loss = F.cross_entropy(logits, torch.tensor([true_label_idx]))
    loss.backward()
    adv = waveform + epsilon * waveform.grad.sign()
    return torch.clamp(adv, -1.0, 1.0).detach()

def pgd_attack(model, waveform, true_label_idx, epsilon, iters=10):
    alpha = epsilon / (iters / 2.0)
    adv = waveform.clone()
    for _ in range(iters):
        adv = adv.clone().requires_grad_(True)
        logits = model(waveform_to_spectrogram(adv.unsqueeze(0)))
        loss = F.cross_entropy(logits, torch.tensor([true_label_idx]))
        loss.backward()
        adv = adv + alpha * adv.grad.sign()
        perturbation = torch.clamp(adv - waveform, -epsilon, epsilon)
        adv = torch.clamp(waveform + perturbation, -1.0, 1.0).detach()
    return adv

@torch.no_grad()
def predict(model, waveform):
    logits = model(waveform_to_spectrogram(waveform.unsqueeze(0)))
    return int(torch.argmax(logits, dim=-1))

def attack_success_rate(model, dataset, attack_fn, epsilon):
    attacked, successes = 0, 0
    for waveform, true_idx in dataset:
        if predict(model, waveform) != true_idx:
            continue
        attacked += 1
        adv = attack_fn(model, waveform, true_idx, epsilon)
        successes += int(predict(model, adv) != true_idx)
    return successes / attacked if attacked else float('nan'), attacked

EVAL_EPSILON = 0.02
results = {}
for name, model in [('Baseline', baseline_model), ('PGD-AT', at_model)]:
    clean_acc = clean_accuracy(model, test_set)
    fgsm_asr, fgsm_n = attack_success_rate(model, test_set, fgsm_attack, EVAL_EPSILON)
    pgd_asr, pgd_n = attack_success_rate(model, test_set, pgd_attack, EVAL_EPSILON)
    results[name] = (clean_acc, fgsm_asr, pgd_asr)
    print(f"{name}: clean {clean_acc*100:.1f}%, FGSM ASR {fgsm_asr*100:.1f}% (n={fgsm_n}), PGD ASR {pgd_asr*100:.1f}% (n={pgd_n})")

Baseline: clean 56.5%, FGSM ASR 68.1% (n=226), PGD ASR 100.0% (n=226)


PGD-AT: clean 48.5%, FGSM ASR 82.0% (n=194), PGD ASR 100.0% (n=194)


### Summary
Honest reading: at this reduced scale (1500 training clips, 5 epochs, 5-step inner PGD), mixed-batch
PGD-AT costs real clean accuracy (56.5% to 48.5%) without buying back any measurable robustness, FGSM
ASR actually rises and PGD ASR stays saturated at 100%. This mirrors the same finding in the vision
repo's `defenses/01_adversarial_training.ipynb` on GTSRB: the literature's PGD-AT results are reported at
a scale (hundreds of epochs, full datasets) far beyond what a from-scratch CPU demo can reach, so a
weak or absent robustness gain here is the expected outcome of that gap, not a broken implementation,
the earlier pure-adversarial attempt that collapsed clean accuracy to 19.8% (see the training cell's
note above) was the actual instability, already fixed by mixing clean and adversarial loss.

In [6]:
print(f"{'Model':<10} {'Clean Acc':>10} {'FGSM ASR':>10} {'PGD ASR':>10}")
for name, (clean_acc, fgsm_asr, pgd_asr) in results.items():
    print(f"{name:<10} {clean_acc*100:>9.1f}% {fgsm_asr*100:>9.1f}% {pgd_asr*100:>9.1f}%")

Model       Clean Acc   FGSM ASR    PGD ASR
Baseline        56.5%      68.1%     100.0%
PGD-AT          48.5%      82.0%     100.0%
